### N-gram language models or how to write scientific papers (4 pts)

We shall train our language model on a corpora of [ArXiv](http://arxiv.org/) articles and see if we can generate a new one!

![img](https://media.npr.org/assets/img/2013/12/10/istock-18586699-monkey-computer_brick-16e5064d3378a14e0e4c2da08857efe03c04695e-s800-c85.jpg)

_data by neelshah18 from [here](https://www.kaggle.com/neelshah18/arxivdataset/)_

_Disclaimer: this has nothing to do with actual science. But it's fun, so who cares?!_

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# Alternative manual download link: https://yadi.sk/d/_nGyU2IajjR9-w
!wget "https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1" -O arxivData.json.tar.gz
!tar -xvzf arxivData.json.tar.gz
data = pd.read_json("./arxivData.json")
data.sample(n=5)

--2026-03-25 21:56:28--  https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.65.18, 2620:100:6021:18::a27d:4112
Connecting to www.dropbox.com (www.dropbox.com)|162.125.65.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/0mulrothty5o8i8ud9gz2/arxivData.json.tar.gz?rlkey=n759u5qx2xpxxglmrl390vwvk&dl=1 [following]
--2026-03-25 21:56:29--  https://www.dropbox.com/scl/fi/0mulrothty5o8i8ud9gz2/arxivData.json.tar.gz?rlkey=n759u5qx2xpxxglmrl390vwvk&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc5c748ed815e47d5eca8a672156.dl.dropboxusercontent.com/cd/0/inline/C9X7o9gFdusIFFmtx8u-FOUKoNIALrY6tdbijRYFrNazvPK1qSh8y4C7SUSDPU32JTxhmNNPgJnE4JgC_Tmewttxi6whJyvT4_STVTxDFn7lTCz-7UZXCuhb7CXibNBSiPc/file?dl=1# [following]
--2026-03-25 21:56:30--  https://uc5c748ed815e47d5eca8a672156.dl.dropbox

,author,day,id,link,month,summary,tag,title,year
20346,"[{'name': 'Dmytro Terletskyi'}, {'name': 'Alex...",14,1510.04194v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",10,This paper contains description of such knowle...,"[{'term': 'cs.AI', 'scheme': 'http://arxiv.org...",Object-Oriented Dynamic Networks,2015
33677,"[{'name': 'Zhaofei Yu'}, {'name': 'Feng Chen'}...",3,1509.00998v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",9,Causal inference in cue combination is to deci...,"[{'term': 'cs.NE', 'scheme': 'http://arxiv.org...",Sampling-based Causal Inference in Cue Combina...,2015
30277,"[{'name': 'Edmar R. S. de Rezende'}, {'name': ...",28,1711.10394v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",11,The recent computer graphics developments have...,"[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Exposing Computer Generated Images by Using De...,2017
35581,"[{'name': 'Pierrick Tranouez'}, {'name': 'Anto...",9,0911.1708v1,"[{'rel': 'related', 'href': 'http://dx.doi.org...",11,In this paper we sum up our works on multiscal...,"[{'term': 'cs.AI', 'scheme': 'http://arxiv.org...",Different goals in multiscale simulations and ...,2009
40041,"[{'name': 'Abhimanyu Dubey'}, {'name': 'Sumeet...",22,1709.07914v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",9,The study of virality and information diffusio...,"[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Modeling Image Virality with Pairwise Spatial ...,2017


In [3]:
# assemble lines: concatenate title and description
lines = data.apply(lambda row: row['title'] + ' ; ' + row['summary'].replace("\n", ' '), axis=1).tolist()

sorted(lines, key=len)[:3]

['Differential Contrastive Divergence ; This paper has been retracted.',
 'What Does Artificial Life Tell Us About Death? ; Short philosophical essay',
 'P=NP ; We claim to resolve the P=?NP problem via a formal argument for P=NP.']

### Tokenization

You know the dril. The data is messy. Go clean the data. Use WordPunctTokenizer or something.


In [4]:
from nltk.tokenize import WordPunctTokenizer
tokenizer = WordPunctTokenizer()
lines = [' '.join(tokenizer.tokenize(line.lower())) for line in lines]

In [5]:
assert sorted(lines, key=len)[0] == \
    'differential contrastive divergence ; this paper has been retracted .'
assert sorted(lines, key=len)[2] == \
    'p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .'

### N-Gram Language Model (1point)

A language model is a probabilistic model that estimates text probability: the joint probability of all tokens $w_t$ in text $X$: $P(X) = P(w_1, \dots, w_T)$.

It can do so by following the chain rule:
$$P(w_1, \dots, w_T) = P(w_1)P(w_2 \mid w_1)\dots P(w_T \mid w_1, \dots, w_{T-1}).$$

The problem with such approach is that the final term $P(w_T \mid w_1, \dots, w_{T-1})$ depends on $n-1$ previous words. This probability is impractical to estimate for long texts, e.g. $T = 1000$.

One popular approximation is to assume that next word only depends on a finite amount of previous words:

$$P(w_t \mid w_1, \dots, w_{t - 1}) = P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1})$$

Such model is called __n-gram language model__ where n is a parameter. For example, in 3-gram language model, each word only depends on 2 previous words.

$$
    P(w_1, \dots, w_n) = \prod_t P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1}).
$$

You can also sometimes see such approximation under the name of _n-th order markov assumption_.

The first stage to building such a model is counting all word occurences given N-1 previous words

In [6]:
from tqdm import tqdm
from collections import defaultdict, Counter

# special tokens:
# - `UNK` represents absent tokens,
# - `EOS` is a special token after the end of sequence

UNK, EOS = "_UNK_", "_EOS_"

def count_ngrams(lines, n):
    """
    Count how many times each word occured after (n - 1) previous words
    :param lines: an iterable of strings with space-separated tokens
    :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

    When building counts, please consider the following two edge cases:
    - if prefix is shorter than (n - 1) tokens, it should be padded with UNK. For n=3,
      empty prefix: "" -> (UNK, UNK)
      short prefix: "the" -> (UNK, the)
      long prefix: "the new approach" -> (new, approach)
    - you should add a special token, EOS, at the end of each sequence
      "... with deep neural networks ." -> (..., with, deep, neural, networks, ., EOS)
      count the probability of this token just like all others.
    """
    counts = defaultdict(Counter)
    # counts[(word1, word2)][word3] = how many times word3 occured after (word1, word2)

    for line in tqdm(lines):
        tokens = [UNK] * (n - 1) + line.split() + [EOS]

        for i in range(n - 1, len(tokens)):
            next_token = tokens[i]
            prefix = tuple(tokens[i - n + 1:i])
            counts[prefix][next_token] += 1

    return counts
    #for line in tqdm(lines):
    #    tokens = line.split()
    #    tokens.append(EOS)

    #    for i in range(len(tokens)):
    #        prefix = tokens[max(0, i - n + 1):i]



In [7]:
# let's test it
dummy_lines = sorted(lines, key=len)[:100]
dummy_counts = count_ngrams(dummy_lines, n=3)
assert set(map(len, dummy_counts.keys())) == {2}, "please only count {n-1}-grams"
assert len(dummy_counts[('_UNK_', '_UNK_')]) == 78
assert dummy_counts['_UNK_', 'a']['note'] == 3
assert dummy_counts['p', '=']['np'] == 2
assert dummy_counts['author', '.']['_EOS_'] == 1

100%|██████████| 100/100 [00:00<00:00, 11159.81it/s]


Once we can count N-grams, we can build a probabilistic language model.
The simplest way to compute probabilities is in proporiton to counts:

$$ P(w_t | prefix) = { Count(prefix, w_t) \over \sum_{\hat w} Count(prefix, \hat w) } $$

In [8]:
class NGramLanguageModel:
    def __init__(self, lines, n):
        """
        Train a simple count-based language model:
        compute probabilities P(w_t | prefix) given ngram counts

        :param n: computes probability of next token given (n - 1) previous words
        :param lines: an iterable of strings with space-separated tokens
        """
        assert n >= 1
        self.n = n

        counts = count_ngrams(lines, self.n)

        # compute token proabilities given counts
        self.probs = defaultdict(Counter)
        # probs[(word1, word2)][word3] = P(word3 | word1, word2)

        # populate self.probs with actual probabilities
        for prefix in counts:
            next_token_counts: dict[str, int] = counts[prefix]
            total_count = sum(next_token_counts.values())

            self.probs[prefix] = {
                next_token: count / total_count
                for next_token, count in next_token_counts.items()
            }

    def get_possible_next_tokens(self, prefix):
        """
        :param prefix: string with space-separated prefix tokens
        :returns: a dictionary {token : it's probability} for all tokens with positive probabilities
        """
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.n + 1):]
        prefix = [ UNK ] * (self.n - 1 - len(prefix)) + prefix
        return self.probs[tuple(prefix)]

    def get_next_token_prob(self, prefix, next_token):
        """
        :param prefix: string with space-separated prefix tokens
        :param next_token: the next token to predict probability for
        :returns: P(next_token|prefix) a single number, 0 <= P <= 1
        """
        return self.get_possible_next_tokens(prefix).get(next_token, 0)

Let's test it!

In [9]:
dummy_lm = NGramLanguageModel(dummy_lines, n=3)

p_initial = dummy_lm.get_possible_next_tokens('') # '' -> ['_UNK_', '_UNK_']
assert np.allclose(p_initial['learning'], 0.02)
assert np.allclose(p_initial['a'], 0.13)
assert np.allclose(p_initial.get('meow', 0), 0)
assert np.allclose(sum(p_initial.values()), 1)

p_a = dummy_lm.get_possible_next_tokens('a') # '' -> ['_UNK_', 'a']
assert np.allclose(p_a['machine'], 0.15384615)
assert np.allclose(p_a['note'], 0.23076923)
assert np.allclose(p_a.get('the', 0), 0)
assert np.allclose(sum(p_a.values()), 1)

assert np.allclose(dummy_lm.get_possible_next_tokens('a note')['on'], 1)
assert dummy_lm.get_possible_next_tokens('a machine') == \
    dummy_lm.get_possible_next_tokens("there have always been ghosts in a machine"), \
    "your 3-gram model should only depend on 2 previous words"

100%|██████████| 100/100 [00:00<00:00, 8185.92it/s]


Now that you've got a working n-gram language model, let's see what sequences it can generate. But first, let's train it on the whole dataset.

In [10]:
lm = NGramLanguageModel(lines, n=3)

100%|██████████| 41000/41000 [00:18<00:00, 2177.71it/s]


The process of generating sequences is... well, it's sequential. You maintain a list of tokens and iteratively add next token by sampling with probabilities.

$ X = [] $

__forever:__
* $w_{next} \sim P(w_{next} | X)$
* $X = concat(X, w_{next})$


Instead of sampling with probabilities, one can also try always taking most likely token, sampling among top-K most likely tokens or sampling with temperature. In the latter case (temperature), one samples from

$$w_{next} \sim {P(w_{next} | X) ^ {1 / \tau} \over \sum_{\hat w} P(\hat w | X) ^ {1 / \tau}}$$

Where $\tau > 0$ is model temperature. If $\tau << 1$, more likely tokens will be sampled with even higher probability while less likely tokens will vanish.

In [26]:
def get_next_token(lm, prefix, temperature=1.0):
    """
    return next token after prefix;
    :param temperature: samples proportionally to lm probabilities ^ (1 / temperature)
        if temperature == 0, always takes most likely token. Break ties arbitrarily.
    """
    #не очень понял тут как работает
    next_token_probs: dict[str, float] = lm.get_possible_next_tokens(prefix)
    tokens, probs = zip(*next_token_probs.items())
    #assert temperature == 1.0, "общибка"
    if temperature == 0.0:
        return tokens[np.argmax(probs)]

    probs = np.log(probs) / temperature
    probs = np.exp(probs)
    probs = probs / probs.sum()

    return np.random.choice(tokens, p=probs)



In [27]:
from collections import Counter
test_freqs = Counter([get_next_token(lm, 'there have') for _ in range(10000)])
assert 250 < test_freqs['not'] < 450
assert 8500 < test_freqs['been'] < 9500
assert 1 < test_freqs['lately'] < 200

test_freqs = Counter([get_next_token(lm, 'deep', temperature=1.0) for _ in range(10000)])
assert 1500 < test_freqs['learning'] < 3000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.5) for _ in range(10000)])
assert 8000 < test_freqs['learning'] < 9000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.0) for _ in range(10000)])
assert test_freqs['learning'] == 10000

print("Looks nice!")

Looks nice!


Let's have fun with this model

In [22]:
prefix = 'artificial' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

artificial neural networks are empirically and find regions of at least as effective feature extraction function for a growing interest in understanding neural networks for structured outputs . to deal with spatially uncertain alarm signals ; this paper , we propose a self - organization , and identify their demographics using automatic differentiation of individuality which is used to solve different problems . _EOS_


In [23]:
prefix = 'bridging the' # <- more of your ideas

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.5)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

bridging the gap of 1 . 48 % is state - of - view stereo quality assurance relative to standard and nonstandard probability spaces containing a high level task , running with a bounded preferential - attachment , and discuss its benefits are two important problems such as trapping in local search for 3d pose estimation systems and for early detection of adverse drug events in the case where the system is 85 , and that of simple genetic algorithm ( the inference time over the output distributions in the proposed stochastic algorithms converge towards nash equilibria . for feature subsets catapults


__More in the homework:__ nucleus sampling, top-k sampling, beam search(not for the faint of heart).

### Evaluating language models: perplexity (1point)

Perplexity is a measure of how well your model approximates the true probability distribution behind the data. __Smaller perplexity = better model__.

To compute perplexity on one sentence, use:
$$
    {\mathbb{P}}(w_1 \dots w_N) = P(w_1, \dots, w_N)^{-\frac1N} = \left( \prod_t P(w_t \mid w_{t - n}, \dots, w_{t - 1})\right)^{-\frac1N},
$$


On the corpora level, perplexity is a product of probabilities of all tokens in all sentences to the power of $1/N$, where $N$ is __total length (in tokens) of all sentences__ in corpora.

This number can quickly get too small for float32/float64 precision, so we recommend you to first compute log-perplexity (from log-probabilities) and then take the exponent.

In [33]:
def perplexity(lm, lines, min_logprob=np.log(10 ** -50.)):
    """
    :param lines: a list of strings with space-separated tokens
    :param min_logprob: if log(P(w | ...)) is smaller than min_logprop, set it equal to min_logrob
    :returns: corpora-level perplexity - a single scalar number from the formula above

    Note: do not forget to compute P(w_first | empty) and P(eos | full_sequence)

    PLEASE USE lm.get_next_token_prob and NOT lm.get_possible_next_tokens
    """
    total_logprob = 0.0
    total_tokens = 0

    for line in lines:
        tokens = line.split()

        # считаем вероятности всех токенов строки + EOS
        for i in range(len(tokens) + 1):
            prefix = ' '.join(tokens[:i])

            if i < len(tokens):
                next_token = tokens[i]
            else:
                next_token = EOS

            prob = lm.get_next_token_prob(prefix, next_token)

            logprob = np.log(prob) if prob > 0 else -np.inf
            logprob = max(logprob, min_logprob)

            total_logprob += logprob
            total_tokens += 1

    return np.exp(-total_logprob / total_tokens)

In [34]:
lm1 = NGramLanguageModel(dummy_lines, n=1)
lm3 = NGramLanguageModel(dummy_lines, n=3)
lm10 = NGramLanguageModel(dummy_lines, n=10)

ppx1 = perplexity(lm1, dummy_lines)
ppx3 = perplexity(lm3, dummy_lines)
ppx10 = perplexity(lm10, dummy_lines)
ppx_missing = perplexity(lm3, ['the jabberwock , with eyes of flame , '])  # thanks, L. Carrol

print("Perplexities: ppx1=%.3f ppx3=%.3f ppx10=%.3f" % (ppx1, ppx3, ppx10))

assert all(0 < ppx < 500 for ppx in (ppx1, ppx3, ppx10)), "perplexity should be non-negative and reasonably small"
assert ppx1 > ppx3 > ppx10, "higher N models should overfit and "
assert np.isfinite(ppx_missing) and ppx_missing > 10 ** 6, "missing words should have large but finite perplexity. " \
    " Make sure you use min_logprob right"
assert np.allclose([ppx1, ppx3, ppx10], (318.2132342216302, 1.5199996213739575, 1.1838145037901249))

100%|██████████| 100/100 [00:00<00:00, 9868.72it/s]

Perplexities: ppx1=318.213 ppx3=1.520 ppx10=1.184


Now let's measure the actual perplexity: we'll split the data into train and test and score model on test data only.

In [35]:
from sklearn.model_selection import train_test_split
train_lines, test_lines = train_test_split(lines, test_size=0.25, random_state=42)

for n in (1, 2, 3):
    lm = NGramLanguageModel(n=n, lines=train_lines)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))


100%|██████████| 30750/30750 [00:03<00:00, 9717.85it/s]


N = 1, Perplexity = 1832.23136


100%|██████████| 30750/30750 [00:07<00:00, 3924.89it/s]


N = 2, Perplexity = 85653987.28774


100%|██████████| 30750/30750 [00:18<00:00, 1695.77it/s]


N = 3, Perplexity = 61999196259043346743296.00000


In [ ]:
# whoops, it just blew up :)
#потому что модель без сглаживания оценивается на test set
#и bigram/trigram массово получают нулевые вероятности для unseen слов
#и контекстов

### LM Smoothing

The problem with our simple language model is that whenever it encounters an n-gram it has never seen before, it assigns it with the probabilitiy of 0. Every time this happens, perplexity explodes.

To battle this issue, there's a technique called __smoothing__. The core idea is to modify counts in a way that prevents probabilities from getting too low. The simplest algorithm here is Additive smoothing (aka [Lapace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing)):

$$ P(w_t | prefix) = { Count(prefix, w_t) + \delta \over \sum_{\hat w} (Count(prefix, \hat w) + \delta) } $$

If counts for a given prefix are low, additive smoothing will adjust probabilities to a more uniform distribution. Not that the summation in the denominator goes over _all words in the vocabulary_.

Here's an example code we've implemented for you:

In [36]:
class LaplaceLanguageModel(NGramLanguageModel):
    """ this code is an example, no need to change anything """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        counts = count_ngrams(lines, self.n)
        self.vocab = set(token for token_counts in counts.values() for token in token_counts)
        self.probs = defaultdict(Counter)

        for prefix in counts:
            token_counts = counts[prefix]
            total_count = sum(token_counts.values()) + delta * len(self.vocab)
            self.probs[prefix] = {token: (token_counts[token] + delta) / total_count
                                          for token in token_counts}
    def get_possible_next_tokens(self, prefix):
        token_probs = super().get_possible_next_tokens(prefix)
        missing_prob_total = 1.0 - sum(token_probs.values())
        missing_prob = missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        return {token: token_probs.get(token, missing_prob) for token in self.vocab}

    def get_next_token_prob(self, prefix, next_token):
        token_probs = super().get_possible_next_tokens(prefix)
        if next_token in token_probs:
            return token_probs[next_token]
        else:
            missing_prob_total = 1.0 - sum(token_probs.values())
            missing_prob_total = max(0, missing_prob_total) # prevent rounding errors
            return missing_prob_total / max(1, len(self.vocab) - len(token_probs))


**Disclaimer**: the implementation above assumes all words unknown within a given context to be equally likely, *as well as the words outside of vocabulary*. Therefore, its' perplexity will be lower than it should when encountering such words. Therefore, comparing it with a model with fewer unknown words will not be fair. When implementing your own smoothing, you may handle this by adding a virtual `UNK` token of non-zero probability. Technically, this will result in a model where probabilities do not add up to $1$, but it is close enough for a practice excercise.

In [37]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = LaplaceLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

100%|██████████| 100/100 [00:00<00:00, 13572.92it/s]


In [38]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=0.1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

100%|██████████| 30750/30750 [00:03<00:00, 9583.60it/s]


N = 1, Perplexity = 1832.66878


100%|██████████| 30750/30750 [00:09<00:00, 3392.54it/s]


N = 2, Perplexity = 470.48021


100%|██████████| 30750/30750 [00:13<00:00, 2200.56it/s]


N = 3, Perplexity = 3679.44765


In [ ]:
# optional: try to sample tokens from such a model

#Проблема обычной n-gram модели в том, что для unseen n-gram она дает
#вероятность 0, из-за чего perplexity резко растет. Поэтому мы юзаем
#additive smoothing: прибавляю маленькое кси к каждому count.
#Так все вероятности остаются ненулевыми и модель работает устойчивее на
#test set.

### Kneser-Ney smoothing (2 points)

Additive smoothing is simple, reasonably good but definitely not a State of The Art algorithm.


Your final task in this notebook is to implement [Kneser-Ney](https://en.wikipedia.org/wiki/Kneser%E2%80%93Ney_smoothing) smoothing.

It can be computed recurrently, for n>1:

$$P_{kn}(w_t | prefix_{n-1}) = { \max(0, Count(prefix_{n-1}, w_t) - \delta) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)} + \lambda_{prefix_{n-1}} \cdot P_{kn}(w_t | prefix_{n-2})$$

where
- $prefix_{n-1}$ is a tuple of {n-1} previous tokens
- $lambda_{prefix_{n-1}}$ is a normalization constant chosen so that probabilities add up to 1
- Unigram $P_{kn}(w_t | prefix_{n-2})$ corresponds to Kneser Ney smoothing for {N-1}-gram language model.
- Unigram $P_{kn}(w_t)$ is a special case: how likely it is to see x_t in an unfamiliar context

See lecture slides or wiki for more detailed formulae.

__Your task__ is to
- implement `KneserNeyLanguageModel` class,
- test it on 1-3 gram language models
- find optimal (within reason) smoothing delta for 3-gram language model with Kneser-Ney smoothing

In [39]:
class KneserNeyLanguageModel(NGramLanguageModel):
    """
    N-граммная языковая модель со сглаживанием Кнезера-Нея (Kneser-Ney).

    Идея:
    - для знакомых n-грамм мы берем их count, но уменьшаем на delta;
    - оставшуюся вероятность не теряем, а "передаем вниз" в модель меньшего порядка;
    - для unigram используем не обычные частоты слов, а continuation counts:
      сколько разных левых контекстов было у слова.
    """

    def __init__(self, lines, n, delta=1.0):
        # порядок модели: 1 для unigram, 2 для bigram, 3 для trigram и т.д.
        self.n = n

        # параметр absolute discounting:
        # именно его мы вычитаем из count(prefix, token)
        self.delta = delta

        # Считаем n-граммы сразу для всех порядков:
        # 1-граммы, 2-граммы, ..., n-граммы.
        # Это нужно, потому что Kneser-Ney рекурсивно откатывается
        # от модели порядка n к модели порядка n-1.
        self.counts_by_order = {
            k: count_ngrams(lines, k) for k in range(1, n + 1)
        }

        # Словарь токенов берем из unigram-частот.
        # Для unigram prefix — пустой tuple: ()
        self.vocab = set(self.counts_by_order[1][()].keys())

        # Для unigram Kneser-Ney нужна не обычная частота слова,
        # а число РАЗНЫХ левых контекстов, в которых это слово встречалось.
        #
        # Например, если слово "cat" встречалось после:
        # "the", "a", "black"
        # то continuation_count["cat"] = 3
        #
        # Это делает Kneser-Ney лучше обычного сглаживания:
        # слово считается "хорошим", если оно встречается
        # после многих разных контекстов, а не просто часто.
        self.continuation_count = Counter()

        # continuation counts удобно считать по bigram-частотам:
        # prefix длины 1 -> какие токены за ним встречались.
        # Для unigram модели bigram'ов нет, поэтому обработаем отдельно.
        bigram_counts = self.counts_by_order[2] if n >= 2 else {}

        # bigram_counts имеет вид:
        # { (prefix_token,) : Counter({next_token: count, ...}), ... }
        #
        # Если token встретился после данного prefix, значит у token есть
        # еще один distinct left context.
        for prefix, next_tokens in bigram_counts.items():
            for token in next_tokens:
                self.continuation_count[token] += 1

        # Общее число различных bigram types:
        # сумма continuation counts по всем токенам.
        #
        # Это знаменатель для unigram Kneser-Ney:
        # P_kn(w) = continuation_count[w] / total_bigram_types
        self.total_bigram_types = sum(self.continuation_count.values())

    def _prepare_prefix(self, prefix, order):
        """
        Приводит prefix к tuple нужной длины: (order - 1).

        Что делает:
        1. если prefix пришел строкой, разбивает его на токены;
        2. если prefix слишком длинный — берет только последние (order - 1) токенов;
        3. если prefix слишком короткий — дополняет слева UNK.

        Примеры для order = 3:
        ""            -> (UNK, UNK)
        "the"         -> (UNK, "the")
        "the cat sat" -> ("cat", "sat")
        """
        # Если prefix — строка, превращаем в список токенов
        if isinstance(prefix, str):
            tokens = prefix.split()
        else:
            # Иначе считаем, что это уже список/tuple токенов
            tokens = list(prefix)

        # Оставляем только последние (order - 1) токенов
        tokens = tokens[max(0, len(tokens) - order + 1):]

        # Если токенов меньше, чем нужно, дополняем слева UNK
        tokens = [UNK] * (order - 1 - len(tokens)) + tokens

        # Возвращаем tuple, потому что именно tuple используется как ключ в counts
        return tuple(tokens)

    def _get_prob(self, prefix, next_token, order):
        """
        Рекурсивно вычисляет вероятность Kneser-Ney:
        P_kn(next_token | prefix) для модели порядка order.

        Здесь prefix должен соответствовать длине (order - 1),
        но на всякий случай мы все равно его нормализуем.
        """
        # БАЗОВЫЙ СЛУЧАЙ: unigram model
        if order == 1:
            # Если почему-то нет bigram types, делаем безопасный fallback:
            # равномерное распределение по словарю
            if self.total_bigram_types == 0:
                return 1.0 / max(len(self.vocab), 1)

            # Unigram Kneser-Ney:
            # вероятность слова пропорциональна числу разных контекстов,
            # в которых это слово встречалось
            return self.continuation_count[next_token] / self.total_bigram_types

        # Нормализуем prefix под текущий order
        prefix = self._prepare_prefix(prefix, order)

        # Берем counts именно для текущего порядка
        # Например, если order = 3, то это trigram counts
        counts = self.counts_by_order[order]

        # Если такого prefix вообще не было в обучении,
        # полностью откатываемся к модели меньшего порядка
        if prefix not in counts:
            # Для backoff убираем самый левый токен prefix:
            # (w_{t-2}, w_{t-1}) -> (w_{t-1},)
            return self._get_prob(prefix[1:], next_token, order - 1)

        # Counter всех токенов, которые встречались после данного prefix
        prefix_counter = counts[prefix]

        # Общее число наблюдений после данного prefix:
        # c(prefix) = sum_w c(prefix, w)
        c_prefix = sum(prefix_counter.values())

        # Count конкретного перехода c(prefix, next_token)
        c_prefix_token = prefix_counter[next_token]

        # Число разных токенов, которые вообще встречались после prefix.
        # Это нужно для lambda(prefix).
        num_types_after_prefix = len(prefix_counter)

        # Первая часть формулы:
        # max(c(prefix, w) - delta, 0) / c(prefix)
        #
        # Мы уменьшаем count на delta, но не даем ему стать отрицательным.
        first_term = max(c_prefix_token - self.delta, 0) / c_prefix

        # Нормировочная константа lambda(prefix):
        # сколько вероятностной массы мы "сняли" с наблюдаемых переходов
        # и теперь должны передать вниз в lower-order model
        lambda_prefix = self.delta * num_types_after_prefix / c_prefix

        # Вероятность из модели меньшего порядка
        lower_prob = self._get_prob(prefix[1:], next_token, order - 1)

        # Полная формула Kneser-Ney:
        # discounted higher-order part + backoff to lower-order model
        return first_term + lambda_prefix * lower_prob

    def get_possible_next_tokens(self, prefix):
        """
        Возвращает словарь:
        {token: P_kn(token | prefix)}
        для всех токенов из словаря.

        Это удобно, если нужно получить всё распределение целиком.
        """
        # Нормализуем prefix под порядок текущей модели
        prefix = self._prepare_prefix(prefix, self.n)

        # Считаем вероятность каждого токена словаря
        return {
            token: self._get_prob(prefix, token, self.n)
            for token in self.vocab
        }

    def get_next_token_prob(self, prefix, next_token):
        """
        Возвращает вероятность одного конкретного токена:
        P_kn(next_token | prefix)

        Именно этот метод обычно нужен для perplexity.
        """
        return self._get_prob(prefix, next_token, self.n)

In [40]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = KneserNeyLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

100%|██████████| 100/100 [00:00<00:00, 19082.37it/s]


In [42]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=1.0)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

100%|██████████| 30750/30750 [00:03<00:00, 9065.24it/s]


N = 1, Perplexity = 54176.00002


100%|██████████| 30750/30750 [00:07<00:00, 3926.59it/s]


N = 2, Perplexity = 404.35703


100%|██████████| 30750/30750 [00:15<00:00, 1935.54it/s]


N = 3, Perplexity = 311.53726
